In [1]:
import pandas as pd

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix



In [2]:
df=pd.read_csv(r"C:\Users\USER\Desktop\5thsem_OJT\project1_diabeties\data\prosessed\fizan_P1W2_features.csv")

In [3]:

from sklearn.metrics import (confusion_matrix,accuracy_score,recall_score,f1_score,roc_auc_score,precision_score)


feature_cols = ["age","pregnancies","glucose","blood_pressure","skin_thickness","insulin","bmi","diabetes_pedigree","bmi_catogory","glucose_band"]
x= pd.get_dummies(df[feature_cols],drop_first=True)
y=df["outcome"]

x_train,x_test,y_train,y_test= train_test_split(
 x,y,test_size=0.2,random_state=42,stratify=y)
scaler = StandardScaler()
x_train_scaled=scaler.fit_transform(x_train)
x_test_scaled=scaler.transform(x_test)

logreg = LogisticRegression(max_iter=1000).fit(x_train_scaled,y_train)
knn = KNeighborsClassifier(n_neighbors=25).fit(x_train_scaled, y_train)
decision_tree = DecisionTreeClassifier(max_depth=3,random_state=42).fit(x_train, y_train)

In [4]:
def score_row(name, pred, scores=None):
    cm = confusion_matrix(y_test, pred)
    tn, fp, fn, tp = cm.ravel()

    r = {
        "model": name,
        "accuracy": round(accuracy_score(y_test, pred), 4),
        "precision": round(precision_score(y_test, pred, zero_division= 0), 4),
        "recall": round(recall_score(y_test, pred), 4),
        "f1": round(f1_score(y_test, pred, zero_division=0), 4),
        "found": int(tp),
        "missed": int(fn),
        "false_alarm": int(fp)
    }

    if scores is not None:
        r["auc"] = round(roc_auc_score(y_test, scores), 4)

    return r

print("train:",x_train.shape,"test:", x_test.shape)
print("diabetic patients in the test set:", int (np.sum(y_test)), "of", len(y_test))
for nm,m,xt in [("LogisticRegression",logreg,x_test_scaled),
                ("decision_tree(d=3)", decision_tree,x_test),
                ("KNN (k=25)", knn, x_test_scaled)]:
    print(f" {nm:22s} accuracy{accuracy_score(y_test,m.predict(xt)):.4f}")

train: (760, 13) test: (190, 13)
diabetic patients in the test set: 55 of 190
 LogisticRegression     accuracy0.7526
 decision_tree(d=3)     accuracy0.7211
 KNN (k=25)             accuracy0.7474


In [5]:
TARGET_RECALL = 0.70
GUARD_F1 = 0.50
BEST_SO_FAR = 0.6182  # Week 5 logistic regression at threshold 0.30
print("PROJECT METRIC : recall (guard: F1 >=", GUARD_F1, ")")
print("TARGET          : recall >=", TARGET_RECALL)
print("BEST SO FAR     :", BEST_SO_FAR,
      "(Week 5 baseline at threshold 0.30)")
print("GAP TO CLOSE    :", round(TARGET_RECALL - BEST_SO_FAR, 4))

PROJECT METRIC : recall (guard: F1 >= 0.5 )
TARGET          : recall >= 0.7
BEST SO FAR     : 0.6182 (Week 5 baseline at threshold 0.30)
GAP TO CLOSE    : 0.0818


### SVM — Simple Definition

**Support Vector Machine (SVM)** is a machine learning algorithm that **separates data into classes by finding the best boundary (hyperplane)** between them.

* Decision Boundary / Hyperplane: Separates the classes.
* Margin: Distance between the boundary and the closest points.
* Support Vectors: Closest points that help define the boundary.

**Simple example:**

> SVM finds the **best boundary** that separates two groups of data with the **largest possible gap (margin)** between them.


In [6]:
from sklearn.svm import SVC

svm_rbf = SVC(random_state=42)
svm_rbf.fit(x_train_scaled,y_train)
pred_rbf = svm_rbf.predict(x_test_scaled)
print("fitted:", svm_rbf)
print("patients flagged:", int (pred_rbf.sum()),"of",len(pred_rbf))

fitted: SVC(random_state=42)
patients flagged: 19 of 190


In [7]:
row_rbf = score_row("SVC rbc (default)", pred_rbf,
                    svm_rbf.decision_function(x_test_scaled))
print(pd.DataFrame([row_rbf]).to_string(index=False))
print()
print("target recall:", TARGET_RECALL, "| best so far:", BEST_SO_FAR)
print("this model  :", row_rbf["recall"])

            model  accuracy  precision  recall     f1  found  missed  false_alarm    auc
SVC rbc (default)    0.7368     0.6316  0.2182 0.3243     12      43            7 0.6378

target recall: 0.7 | best so far: 0.6182
this model  : 0.2182


try straight boundary instead

In [8]:

svm_linear = SVC(kernel="linear", random_state=42)
svm_linear.fit(x_train_scaled, y_train)

pred_linear = svm_linear.predict(x_test_scaled)

row_linear = score_row(
    "SVC Linear",
    pred_linear,
    svm_linear.decision_function(x_test_scaled)
)


print(pd.DataFrame([row_rbf, row_linear]).to_string(index=False))

print()
print("target recall:", TARGET_RECALL, "| best so far:", BEST_SO_FAR)
print("RBF recall   :", row_rbf["recall"])
print("Linear recall:", row_linear["recall"])

            model  accuracy  precision  recall     f1  found  missed  false_alarm    auc
SVC rbc (default)    0.7368     0.6316  0.2182 0.3243     12      43            7 0.6378
       SVC Linear    0.7105     0.0000  0.0000 0.0000      0      55            0 0.6885

target recall: 0.7 | best so far: 0.6182
RBF recall   : 0.2182
Linear recall: 0.0


In [9]:
print("patients the linear SVM flagged as diebetic:",
      int(pred_linear.sum()), "of", len(pred_linear))
print()

print("Linear SVM confusion matrix:")
print(confusion_matrix(y_test, pred_linear))
print("dummy model confusion matrix:")
print(confusion_matrix(y_test, np.zeros_like(y_test)))

print("Linear SVM accuracy:", round(accuracy_score(y_test, pred_linear), 4))

print("Dummy model accuracy:", round(accuracy_score(y_test, np.zeros_like(y_test)), 4))


patients the linear SVM flagged as diebetic: 0 of 190

Linear SVM confusion matrix:
[[135   0]
 [ 55   0]]
dummy model confusion matrix:
[[135   0]
 [ 55   0]]
Linear SVM accuracy: 0.7105
Dummy model accuracy: 0.7105


In [10]:
print("Patients the Linear SVM flagged as diabetic:",
      int(pred_linear.sum()), "of", len(pred_linear))

print()

print("Linear SVM confusion matrix:")
print(confusion_matrix(y_test, pred_linear))

print("Dummy model confusion matrix:")
print(confusion_matrix(y_test, np.zeros_like(y_test)))

print()

print("Linear SVM accuracy:",
      round(accuracy_score(y_test, pred_linear), 4))

print("Dummy model accuracy:",
      round(accuracy_score(y_test, np.zeros_like(y_test)), 4))

Patients the Linear SVM flagged as diabetic: 0 of 190

Linear SVM confusion matrix:
[[135   0]
 [ 55   0]]
Dummy model confusion matrix:
[[135   0]
 [ 55   0]]

Linear SVM accuracy: 0.7105
Dummy model accuracy: 0.7105


Iearn what cIass weight does

Class Weight

Class weight tells the model how much importance to give to each class during training. Using class_weight="balanced" gives more weight to the minority class, helping the model pay more attention to less common cases.

SVC(kernel="linear", class_weight="balanced")

In [22]:
svm_rbf_bal = SVC(random_state=42 , class_weight='balanced').fit(x_train_scaled,y_train)
pred_rbf_bal = svm_rbf_bal.predict(x_test_scaled)
svm_lin_bal = SVC(kernel='linear' , random_state=42, class_weight='balanced').fit(x_train_scaled, y_train) 
pred_lin_bal = svm_lin_bal.predict(x_test_scaled)


row_linear_bal = score_row('SVC linear balanced', pred_lin_bal,svm_linear.decision_function(x_test_scaled))
row_rbf_bal = score_row('SVC rbf balanced ', pred_rbf_bal,svm_rbf.decision_function(x_test_scaled))

print(pd.DataFrame([row_linear,row_linear_bal,row_rbf,row_rbf_bal]).to_string(index=False))

              model  accuracy  precision  recall     f1  found  missed  false_alarm    auc
         SVC Linear    0.7105     0.0000  0.0000 0.0000      0      55            0 0.6885
SVC linear balanced    0.6579     0.4375  0.6364 0.5185     35      20           45 0.6885
  SVC rbc (default)    0.7368     0.6316  0.2182 0.3243     12      43            7 0.6378
  SVC rbf balanced     0.6421     0.4198  0.6182 0.5000     34      21           47 0.6378


In [25]:
scores_bal= svm_rbf_bal.decision_function(x_test_scaled)
print("decision_function output for the first 8 test patients:")
print(np.round(scores_bal[:8], 3))

print("negative = healthy side of the boundary, positive = diabetic side")

print()

print("AUC from decision_function:", 
      round(roc_auc_score(y_test, scores_bal), 4))

print("AUC of the linear SVM that found nobody:",
      round(roc_auc_score(y_test, svm_linear.decision_function(x_test_scaled)), 4))

decision_function output for the first 8 test patients:
[ 0.167 -1.654 -0.666 -0.648 -0.782 -1.159  0.732  1.104]
negative = healthy side of the boundary, positive = diabetic side

AUC from decision_function: 0.6803
AUC of the linear SVM that found nobody: 0.6885


In [40]:
print("Support vectors per class (RBF default):", svm_rbf.n_support_)

print("Total support vectors:", svm_rbf.n_support_.sum())

print(svm_rbf.n_support_, svm_rbf.n_support_.sum(), "of", len(y_train))

print(f"percentage := {svm_rbf.n_support_.sum() / len(y_train) * 100:.1f}% of the training set")

Support vectors per class (RBF default): [276 210]
Total support vectors: 486
[276 210] 486 of 760
percentage := 63.9% of the training set


In [43]:

svm_c10 = SVC(random_state=42, C=10).fit(x_train_scaled, y_train)

print(pd.DataFrame([
    row_rbf,
    score_row(
        "SVC rbf C=10",
        svm_c10.predict(x_test_scaled),
        svm_c10.decision_function(x_test_scaled)
    )
]).to_string(index=False))


            model  accuracy  precision  recall     f1  found  missed  false_alarm    auc
SVC rbc (default)    0.7368     0.6316  0.2182 0.3243     12      43            7 0.6378
     SVC rbf C=10    0.6789     0.4211  0.2909 0.3441     16      39           22 0.6249
